# Results & Retrospective
**FULL WRITE-UP · TRUE-HOLDOUT CONFIRMATION · LESSONS LEARNED**

---

## Content

- [1 · Objective](#1-objective)
- [2 · The modeling journey](#2-the-modeling-journey)
- [3 · Results — internal held-out test](#3-results-internal-held-out-test)
- [4 · Results — true AIM holdout](#4-results-true-aim-holdout)
- [5 · Data-leakage investigation](#5-data-leakage-investigation)
- [6 · Root cause — why the original score was 0.28](#6-root-cause-why-the-original-score-was-028)
- [7 · Lessons learned](#7-lessons-learned)
- [8 · Recommendations](#8-recommendations)

**Headline:** on the hidden out-of-sample set, the tuned champion reaches **bad-buy F1 0.409**
— clearing the original assessment's 0.40 bar. The project was historically marked as a failure not
because of the model, but because the **baseline was submitted at the default threshold**. A
rigorous audit ([`docs/DATA_LEAKAGE_AUDIT.md`](../docs/DATA_LEAKAGE_AUDIT.md)) shows no data leakage
and no overfitting — the modeling was sound, the last-mile submission was not.

This notebook is the narrative spine of the project: it consolidates the modeling journey, the
final numbers (internal test *and* true holdout), the root-cause of the original 0.28 score, and
the lessons learned — replacing the former `docs/RESULTS.md`.

## 1 · Objective

Predict, **before purchase**, whether a used car bought at auction will be a "Bad Buy" (a
lemon that can't be resold). The target `IsBadBuy` is strongly imbalanced, so accuracy is
meaningless and the project optimizes the **F1 score of the bad-buy class**. The assessment set a
bar of **F1 > 0.40** on a hidden scoring set (`features_aim.csv`).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, io, contextlib
from us_used_vehicle_resales.cleaning import clean_data
from us_used_vehicle_resales.features import engineer_features
from us_used_vehicle_resales.config_features_catalog import features_catalog
from us_used_vehicle_resales.config_models_catalog import models_catalog
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)

INT='../data/02_interim/'; RAW='../data/01_raw/'
pd.options.display.float_format='{:.4f}'.format

def _quiet(fn,*a,**k):
    with contextlib.redirect_stdout(io.StringIO()): return fn(*a,**k)

def load_prep(name):
    X=pd.read_parquet(f'{INT}features_{name}.parquet')
    y=pd.read_parquet(f'{INT}target_{name}.parquet').iloc[:,0]
    Xf=_quiet(engineer_features,_quiet(clean_data,X),print_status=False)
    return Xf, y.loc[Xf.index]

X_train,y_train=load_prep('train')
X_test, y_test =load_prep('test')

facts = pd.Series({
    'Training rows': len(X_train)+len(X_test),
    'Training columns (raw)': 33,
    'Bad-buy rate': f'{pd.concat([y_train,y_test]).mean():.2%}',
    'Held-out test rows': len(X_test),
    'Scoring set (AIM) rows': 7292,
    'Assessment bar': 'F1 > 0.40',
})
facts.to_frame('value')

,value
Training rows,65620
Training columns (raw),33
Bad-buy rate,12.35%
Held-out test rows,13124
Scoring set (AIM) rows,7292
Assessment bar,F1 > 0.40


## 2 · The modeling journey

1. **Baseline** — Logistic Regression on 8 hand-picked features (age, odometer, price anchors,
   two engineered ratios, auction, make). Establishes the floor.
2. **Systematic benchmark** — a self-built **`ModelTracker`** ran **448 experiments** across
   feature sets × model families (Logistic Regression ridge/lasso/elastic-net, Random Forest
   shallow/deep, HistGradientBoosting standard/aggressive), logging F1/recall/precision/ROC-AUC per
   run, flagging the best, and exporting the fitted pipeline. See
   [`05_experiment_framework.ipynb`](05_experiment_framework.ipynb) for the catalogs and tracker
   themselves — the infrastructure that made a fast, repeatable comparison possible instead of
   ad-hoc training runs.
3. **Champion** — **Logistic Regression with an L1 penalty** (`class_weight='balanced'`) on the
   full `all_in_with_noise` feature set. Two findings drove the choice:
   - **Categorical signal is essential:** dropping the categorical/engineered features collapses F1
     from ~0.38 to ~0.29 (see [`06_error_analysis.ipynb`](06_error_analysis.ipynb) for how the model
     leans on `WheelType`). Market-price numerics alone are not enough.
   - **Consistent high recall (~0.60)** at modest precision — the right trade-off for a *triage*
     use case where missing a bad buy is costlier than a false alarm.
4. **Threshold tuning** — with balanced class weights the default 0.5 threshold over-flags. The
   F1-optimal operating point on the held-out test is **threshold ≈ 0.65**.

**A judgment call that aged well:** several Random Forest runs in the tracking log showed F1 > 0.40,
but with recall ~0.3 and precision ~0.7 — useless for a triage filter that must *catch* bad buys.
These were correctly rejected in favour of the higher-recall Logistic Regression.

## 3 · Results — internal held-out test

All models evaluated on the **same held-out test set** (n = 13,124), threshold 0.5 unless
noted. This is the saved output of [`04_evaluation.ipynb`](04_evaluation.ipynb), the results
single-source-of-truth — loaded here, not recomputed, so the two notebooks can never disagree.

In [2]:
internal = pd.read_csv('../data/04_models/model_results_final_test.csv')
internal = internal.sort_values('F1', ascending=False).reset_index(drop=True)
internal

,Model,Features,Recall,Precision,F1,ROC_AUC
0,"LogReg Lasso (L1, balanced)",27,0.6039,0.2696,0.3728,0.7650
1,"Random Forest (deep, balanced)",27,0.6416,0.2425,0.3520,0.7491
2,Baseline LogReg (8 feat),8,0.6114,0.1874,0.2868,0.6662


In [3]:
feats=[f for f in features_catalog['all_in_with_noise'] if f in X_train.columns]
num=[f for f in feats if str(X_train[f].dtype).startswith(('int','float'))]
cat=[f for f in feats if f not in num]
pre=ColumnTransformer([('num',StandardScaler(),num),
                       ('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
champ_pipe=Pipeline([('pre',pre),('clf',models_catalog['log_reg_lasso'])])
champ_pipe.fit(X_train[feats],y_train)
champ_proba_test=champ_pipe.predict_proba(X_test[feats])[:,1]

ths=np.linspace(0.2,0.85,66)
f1s=[f1_score(y_test,(champ_proba_test>=t).astype(int)) for t in ths]
best_t=float(ths[int(np.argmax(f1s))])
pred_t=(champ_proba_test>=best_t).astype(int)
print(f'Tuned threshold {best_t:.2f} -> F1 {f1_score(y_test,pred_t):.4f} '
      f'| Precision {precision_score(y_test,pred_t):.4f} | Recall {recall_score(y_test,pred_t):.4f}')

Tuned threshold 0.65 -> F1 0.4227 | Precision 0.4525 | Recall 0.3967


The internal test confirms: **LogReg Lasso is the strongest model**, and tuning the threshold
lifts it from F1 0.37 to F1 0.42 — the biggest single lever in this project.

## 4 · Results — true AIM holdout

The hidden `target_aim.csv` was later recovered, so the models can be scored on the **true
out-of-sample labels** (7,292 vehicles, 863 bad buys) instead of just the internal test. Both the
baseline and the champion are retrained here (same catalogs, same `random_state=42`) and scored
against the raw `features_aim.csv` end to end — nothing here is copy-pasted from a report.

`target_aim.csv` itself is excluded from the repository (assessment ground truth) and is only
present locally.

In [4]:
base_feats=[f for f in features_catalog['baseline'] if f in X_train.columns]
base_num=[f for f in base_feats if str(X_train[f].dtype).startswith(('int','float'))]
base_cat=[f for f in base_feats if f not in base_num]
base_pre=ColumnTransformer([('num',StandardScaler(),base_num),
                            ('cat',OneHotEncoder(handle_unknown='ignore'),base_cat)])
base_pipe=Pipeline([('pre',base_pre),
                    ('clf',LogisticRegression(class_weight='balanced',random_state=42,max_iter=2000))])
base_pipe.fit(X_train[base_feats],y_train)

aim=pd.read_csv(f'{RAW}features_aim.csv',sep=';')
aim_f=_quiet(engineer_features,_quiet(clean_data,aim),print_status=False)
y_aim=pd.read_csv(f'{RAW}target_aim.csv').iloc[:,0]

def align(feat_list):
    out=aim_f.copy()
    for c in feat_list:
        if str(X_train[c].dtype).startswith(('int','float')):
            out[c]=pd.to_numeric(out[c],errors='coerce').fillna(0)
        else:
            out[c]=out[c].astype(str)
    return out[feat_list]

base_pred_aim=base_pipe.predict(align(base_feats))
base_f1_aim=f1_score(y_aim,base_pred_aim)
base_cm_aim=confusion_matrix(y_aim,base_pred_aim)
print(f'Baseline on AIM -> F1 {base_f1_aim:.4f}  confusion matrix {base_cm_aim.tolist()}')
print('Examiner (original, from the historical scoring report): F1 0.2810, confusion matrix [[4032, 2397], [330, 533]]')

Baseline on AIM -> F1 0.2799  confusion matrix [[4029, 2400], [332, 531]]
Examiner (original, from the historical scoring report): F1 0.2810, confusion matrix [[4032, 2397], [330, 533]]


**Pipeline-fidelity check:** the locally reproduced baseline matches the original examiner's
score almost exactly (±3 rows out of 7,292) — proof that this is the same pipeline the assessment
ran, so every champion number below is trustworthy.

In [5]:
champ_proba_aim=champ_pipe.predict_proba(align(feats))[:,1]

rows=[]
for t in [0.5, best_t]:
    pred=(champ_proba_aim>=t).astype(int)
    rows.append(dict(Threshold=t, F1=f1_score(y_aim,pred), Precision=precision_score(y_aim,pred),
                     Recall=recall_score(y_aim,pred), Flagged=pred.mean()))
champ_aim_results=pd.DataFrame(rows)
champ_aim_results

,Threshold,F1,Precision,Recall,Flagged
0,0.5000,0.3600,0.2595,0.5875,0.2680
1,0.6500,0.4089,0.4522,0.3731,0.0976


In [6]:
gap = internal.loc[internal.Model.str.contains('Lasso'), 'F1'].iloc[0] - champ_aim_results.loc[champ_aim_results.Threshold==0.5,'F1'].iloc[0]
print(f'Internal test F1 (0.5) vs AIM F1 (0.5) gap: {gap:.3f}')
print(f'Champion clears the 0.40 bar at the tuned threshold: '
      f'{champ_aim_results.loc[champ_aim_results.Threshold==best_t,"F1"].iloc[0]:.3f} > 0.40')

Internal test F1 (0.5) vs AIM F1 (0.5) gap: 0.013
Champion clears the 0.40 bar at the tuned threshold: 0.409 > 0.40


**The internal-test F1 and the AIM F1 at threshold 0.5 differ by only ~0.01** — the model
generalizes almost perfectly. **At the tuned threshold the champion clears the 0.40 bar on the real
holdout.** This 0.01 gap is also the strongest empirical evidence against leakage or overfitting:
an inflated internal score would show a much larger gap to the true holdout.

## 5 · Data-leakage investigation

Because the assessment bar wasn't met historically, data leakage was suspected and the
project was shelved. A reproducible audit
([`docs/DATA_LEAKAGE_AUDIT.md`](../docs/DATA_LEAKAGE_AUDIT.md)) checked every vector:

- **Split before feature engineering**, transformers fit on **train only**, **no target-derived
  features**, **zero duplicate rows** across the split → no classic leakage.
- **High-cardinality memorization tested directly:** removing 2,082 memorizable category levels
  (ZIP, model, sub-model) changes test F1 by only **0.003** → no overfitting.
- **Confirmed empirically above:** the ~0.01 internal→holdout gap on true out-of-sample data is the
  final proof that internal scores were never inflated.

**There was no leakage.**

## 6 · Root cause — why the original score was 0.28

The historical failure was a **last-mile submission mistake**, not a modeling defect. Two
compounding errors:

| # | Mistake | Effect on F1 |
|:--|:--------|:-------------|
| 1 | **Baseline model submitted** instead of the LogReg Lasso champion | ~0.28 instead of ~0.36 |
| 2 | **Default threshold 0.5** instead of the tuned 0.65 | ~0.36 instead of ~0.41 |

Either fix alone would have moved the needle; both together are the difference between "failed" and
"0.409, passed". The modeling analysis had already identified the right champion — it simply didn't
make it into the exported predictions file, and the threshold was never applied to the submission.

The more expensive error was the **misdiagnosis afterwards**: concluding "leakage" and abandoning
the project, when the real cause was a cheap process slip.

## 7 · Lessons learned

1. **Threshold tuning is part of the deliverable, not an afterthought.** On an imbalanced
   target, the operating point is a first-class modeling decision — the single biggest lever here
   (0.36 → 0.41).
2. **Evaluate the *final chosen* model on one clean held-out test before shipping.** One number, one
   source of truth. The original confusion ("is RF the best?") came from comparing validation scores
   across several tracking files instead of one final test — see `04_evaluation.ipynb`.
3. **A high F1 with unusable recall is not a win.** Reading the full precision/recall picture, not
   the headline metric, was the right instinct — keep doing that.
4. **When a result disappoints, check the pipeline's last mile before blaming the data.** Verify
   *which* artifact was submitted before reaching for "leakage".
5. **The self-built `ModelTracker` was a genuinely good instinct** — systematic, logged, exportable.
   For production, know that MLflow / Weights & Biases do this off-the-shelf, so the trade-off can be
   named explicitly. The gap wasn't the tool; it was the missing single-source-of-truth test pass.

## 8 · Recommendations

- **Deploy the tuned LogReg Lasso (threshold ≈ 0.65)** as a **triage filter**: it flags ~10 %
  of an unlabeled batch at ~0.45 precision for human review — not an automatic reject.
- Treat a missing `WheelType` (`WheelType = Unknown`) as a **first-order risk flag at intake** — the
  single strongest predictor — but pair it with a second signal for the blind spot identified in
  [`06_error_analysis.ipynb`](06_error_analysis.ipynb): newer, pricier bad buys that don't carry that
  flag.
- Fold the two batch-level statistics (median imputation, `feat_price_cat` quantile bins) into the
  fitted pipeline so single-record scoring is production-safe (audit §2.5).

---

### Reproducibility

| Artifact | Location |
|:---------|:---------|
| Results SSoT (internal test numbers) | [`04_evaluation.ipynb`](04_evaluation.ipynb) |
| Error analysis (confusion matrix, FN/FP segments) | [`06_error_analysis.ipynb`](06_error_analysis.ipynb) |
| Engineering showcase (catalogs, `ModelTracker`) | [`05_experiment_framework.ipynb`](05_experiment_framework.ipynb) |
| Leakage audit | [`docs/DATA_LEAKAGE_AUDIT.md`](../docs/DATA_LEAKAGE_AUDIT.md) |
| Model comparison / threshold / feature-importance charts | [`public/img/`](../public/img/) |

<sub>The hidden `target_aim.csv` is kept out of the repository (assessment ground truth); the AIM
numbers in §4 are computed live in this notebook against it.</sub>